# اليوم الأول — معالجة النصوص والترميز
## Day 1 — Text Processing & Tokenisation

**المدربة / Instructor:** ميعاد المري — Meaad Al-Marri  
**المسار / Track:** Core → Explore → Distinction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/almiyead-rgb/bayan-applied-nlp-course/blob/main/notebooks/01_text_processing_tokenization.ipynb)

> السؤال المحوري: كيف يتحول نص عربي أو إنجليزي إلى تمثيل عددي صالح للنموذج دون إفساد معناه؟
>
> Driving question: How does Arabic or English text become a model-ready numeric representation without damaging its meaning?

**بنهاية المختبر ستتمكن من / By the end, you can:**

1. فحص Unicode بدل التخمين — inspect Unicode rather than guess.
2. بناء نسختين: نص خام محفوظ ونص مهيأ للنموذج — keep raw and model-ready copies.
3. قياس **خصوبة الترميز Token Fertility** ومعدل القطع — measure fertility and truncation.
4. تحويل Token IDs إلى Embeddings بسيطة — map token IDs to simple embeddings.
5. حفظ دليل إنجاز قابل للتسليم عبر GitHub — save GitHub-ready evidence.

## 0) طريقة العمل / How to work

- نفّذ الخلايا بالترتيب: **Runtime → Core → Checkpoint**.
- نتيجة كل خلية موضحة بجملة تبدأ بـ **المتوقع / Expected**.
- إذا تعذر تنزيل نموذج Hugging Face، أكمل بالمرمّز المحلي؛ فهو جزء أساسي يعمل دون شبكة بعد تثبيت الحزم.
- لا تستخدم بيانات حقيقية أو أسماء أو أرقام أشخاص. البيانات أدناه اصطناعية.

**المتوقع / Expected:** في النهاية تظهر العبارة `DAY1_NOTEBOOK1_CORE=PASS`.

In [8]:
# إعداد بيئة قابلة لإعادة التشغيل / Reproducible setup
import importlib.util
import subprocess
import sys

PINNED = {"transformers": "5.15.1", "tokenizers": "0.22.2", "spacy": "3.8.7"}
missing = [name for name in PINNED if importlib.util.find_spec(name) is None]
if missing:
    packages = [f"{name}=={PINNED[name]}" for name in missing]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

Python: 3.13.15
Environment ready / البيئة جاهزة


### لماذا نثبت نسخًا محددة؟ / Why pin versions?

لتقليل اختلاف النتائج بين أجهزة المتدربين. التثبيت مجاني ولا يحتاج اشتراكًا؛ يحتاج اتصالًا بالإنترنت أول مرة فقط. Google Colab المجاني يكفي لهذا اليوم، ولا يضمن GPU أو مدة جلسة ثابتة.

**المتوقع / Expected:** إصدار Python ثم رسالة نجاح البيئة.

In [9]:
import html
import re
import unicodedata
from dataclasses import dataclass
from typing import Iterable

import numpy as np
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing

SEED = 42
rng = np.random.default_rng(SEED)
samples = [
    "أهلاً وسهلاً بكم في برنامج بيان!",
    "مرحبــاً\u00a0بكم",
    "Contact us at learner@example.org",
    "للتجربة فقط: 0551234567",
    "Natural language processing connects text and models.",
]
print("Synthetic samples:", len(samples))

Synthetic samples: 5


## 1) Unicode قبل التنظيف / Unicode before cleaning

الحرف الظاهر قد يتكون من نقطة ترميز واحدة أو أكثر. لا تخلط بين:

- **Unicode code point:** رقم معياري للحرف.
- **UTF-8:** تحويل نقاط الترميز إلى bytes للتخزين والنقل.
- **Token ID:** رقم داخل قاموس مرمّز محدد، وليس رقم Unicode.

نستخدم **NFC** لتوحيد التمثيلات المتكافئة غالبًا، لكن لا نحول العربية بقوة من دون سبب متعلق بالمهمة.

In [10]:
def inspect_unicode(text: str) -> list[dict[str, str]]:
    return [
        {"char": ch, "code_point": f"U+{ord(ch):04X}", "name": unicodedata.name(ch, "UNKNOWN")}
        for ch in text
    ]

unicode_rows = inspect_unicode("أé")
for row in unicode_rows:
    print(row)
assert unicode_rows[0]["code_point"] == "U+0623"
print("Unicode inspection=PASS")

{'char': 'أ', 'code_point': 'U+0623', 'name': 'ARABIC LETTER ALEF WITH HAMZA ABOVE'}
{'char': 'é', 'code_point': 'U+00E9', 'name': 'LATIN SMALL LETTER E WITH ACUTE'}
Unicode inspection=PASS


**المتوقع / Expected:** صف لكل حرف يحوي الشكل و`U+....` والاسم، ثم رسالة نجاح.

### قرار عربي محافظ / Conservative Arabic decision

ابدأ بأقل تعديل ممكن. إزالة التشكيل أو توحيد الألف قد يفيد البحث أو التصنيف، لكنه قد يضر مهامًا تعتمد الشكل الأصلي. لذلك نسجل الإعدادات ولا نطبقها بصمت.

In [11]:
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"
WHITESPACE = re.compile(r"\s+")
HTML_TAG = re.compile(r"<[^>]+>")
EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
SAUDI_MOBILE = re.compile(r"(?<!\d)(?:\+?966|00966|0)?5\d{8}(?!\d)")

@dataclass(frozen=True)
class TextRecord:
    raw_text: str
    model_text: str


def mask_pii(text: str) -> str:
    return SAUDI_MOBILE.sub("<PHONE>", EMAIL.sub("<EMAIL>", text))


def prepare_text(text: str, *, remove_diacritics: bool = False, normalize_alef: bool = False) -> TextRecord:
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    raw = text
    model = unicodedata.normalize("NFC", html.unescape(text))
    model = HTML_TAG.sub(" ", model).replace(TATWEEL, "")
    if remove_diacritics:
        model = ARABIC_DIACRITICS.sub("", model)
    if normalize_alef:
        model = re.sub(r"[إأآٱ]", "ا", model)
    model = WHITESPACE.sub(" ", mask_pii(model)).strip()
    return TextRecord(raw_text=raw, model_text=model)

records = [prepare_text(text) for text in samples]
for record in records:
    print({"raw": record.raw_text, "model": record.model_text})
assert records[1].raw_text != records[1].model_text
assert "learner@example.org" not in records[2].model_text
assert "0551234567" not in records[3].model_text
assert records[2].raw_text == samples[2]
print("Two-copy preprocessing contract=PASS")

{'raw': 'أهلاً وسهلاً بكم في برنامج بيان!', 'model': 'أهلاً وسهلاً بكم في برنامج بيان!'}
{'raw': 'مرحبــاً\xa0بكم', 'model': 'مرحباً بكم'}
{'raw': 'Contact us at learner@example.org', 'model': 'Contact us at <EMAIL>'}
{'raw': 'للتجربة فقط: 0551234567', 'model': 'للتجربة فقط: <PHONE>'}
{'raw': 'Natural language processing connects text and models.', 'model': 'Natural language processing connects text and models.'}
Two-copy preprocessing contract=PASS


**المتوقع / Expected:** تبقى `raw` كما دخلت، بينما تحذف الكشيدة وتوحد المسافات وتستبدل البريد والهاتف بعلامات عامة.

> **قاعدة البيانات:** النسخة الخام للتدقيق وإعادة المعالجة، والنسخة المهيأة لدخول النموذج. لا تُنشر النسخة الخام إن احتوت بيانات شخصية.

### جرّب / Try

غيّر `remove_diacritics` و`normalize_alef` على مثال اصطناعي، ثم اكتب هل التغيير يخدم مهمتك أم يخفي فرقًا مهمًا.

## 2) spaCy كمرحلة قابلة للاختبار / A testable spaCy stage

التقسيم إلى جمل لا يُنفذ بـ`split('.')`. نمرر **نسخة النموذج المحمية** إلى spaCy، ونوثق قيدًا معروفًا: الاختصارات مثل `د.` تحتاج استثناءً أو segmenter مدربًا قبل الإنتاج. هذا المثال يستخدم `sentencizer` القاعدي بلا تنزيل نموذج لغوي.

In [12]:
import spacy

sentence_nlp = spacy.blank("xx")
sentence_nlp.add_pipe("sentencizer")

def split_model_sentences(text: str) -> list[str]:
    protected = prepare_text(text).model_text
    return [sentence.text.strip() for sentence in sentence_nlp(protected).sents if sentence.text.strip()]

sentence_examples = {
    "ar": "الخدمة جيدة. لم يصل الرمز!",
    "en": "The portal stopped. Please retry.",
}
sentence_splits = {language: split_model_sentences(text) for language, text in sentence_examples.items()}
for language, sentences in sentence_splits.items():
    print(language, sentences)

abbreviation_probe = split_model_sentences("راجع د. أحمد. ثم أعد المحاولة.")
print("Known abbreviation probe / حالة تحتاج قاعدة إضافية:", abbreviation_probe)
assert sentence_splits["ar"] == ["الخدمة جيدة.", "لم يصل الرمز!"]
assert sentence_splits["en"] == ["The portal stopped.", "Please retry."]
print("SPACY_SENTENCE_PIPELINE=PASS")

ar ['الخدمة جيدة.', 'لم يصل الرمز!']
en ['The portal stopped.', 'Please retry.']
Known abbreviation probe / حالة تحتاج قاعدة إضافية: ['راجع د.', 'أحمد.', 'ثم أعد المحاولة.']
SPACY_SENTENCE_PIPELINE=PASS


## 2) الترميز إلى كلمات فرعية / Subword tokenisation

المرمّز المحلي التالي **تعليمي ومحدود**: يوضح الرموز الخاصة و`[UNK]` وToken IDs دون تنزيل نموذج. ليس بديلًا عن مرمّز النموذج النهائي. في Explore سنقارن اختياريًا مع mBERT.

In [13]:
special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
known_tokens = [
    "أهلاً", "وسهلاً", "بكم", "في", "برنامج", "بيان", "!", "مرحبا",
    "Contact", "us", "at", "<", "EMAIL", ">", "Natural", "language",
    "processing", "connects", "text", "and", "models", ".",
]
vocab_tokens = special_tokens + known_tokens
vocab = {token: idx for idx, token in enumerate(vocab_tokens)}
local_tokenizer = Tokenizer(WordPiece(vocab=vocab, unk_token="[UNK]"))
local_tokenizer.pre_tokenizer = BertPreTokenizer()
local_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[("[CLS]", vocab["[CLS]"]), ("[SEP]", vocab["[SEP]"])],
)
for record in records:
    encoded = local_tokenizer.encode(record.model_text)
    print(record.model_text)
    print("tokens:", encoded.tokens)
    print("ids:   ", encoded.ids)
assert local_tokenizer.encode("مرحبا بكم").tokens == ["[CLS]", "مرحبا", "بكم", "[SEP]"]
print("Local WordPiece demonstration=PASS")

أهلاً وسهلاً بكم في برنامج بيان!
tokens: ['[CLS]', 'أهلاً', 'وسهلاً', 'بكم', 'في', 'برنامج', 'بيان', '!', '[SEP]']
ids:    [2, 5, 6, 7, 8, 9, 10, 11, 3]
مرحباً بكم
tokens: ['[CLS]', '[UNK]', 'بكم', '[SEP]']
ids:    [2, 1, 7, 3]
Contact us at <EMAIL>
tokens: ['[CLS]', 'Contact', 'us', 'at', '<', 'EMAIL', '>', '[SEP]']
ids:    [2, 13, 14, 15, 16, 17, 18, 3]
للتجربة فقط: <PHONE>
tokens: ['[CLS]', '[UNK]', '[UNK]', '[UNK]', '<', '[UNK]', '>', '[SEP]']
ids:    [2, 1, 1, 1, 16, 1, 18, 3]
Natural language processing connects text and models.
tokens: ['[CLS]', 'Natural', 'language', 'processing', 'connects', 'text', 'and', 'models', '.', '[SEP]']
ids:    [2, 19, 20, 21, 22, 23, 24, 25, 26, 3]
Local WordPiece demonstration=PASS


**المتوقع / Expected:** Tokens وIDs لكل نص. ظهور `[UNK]` يعني أن قاموس هذا المرمّز التعليمي لا يعرف الجزء؛ لا يعني أن النص خاطئ.

### مقياسان قبل اختيار المرمّز / Two pre-selection metrics

\[
\text{Token Fertility}=\frac{\text{عدد الرموز الفرعية}}{\text{عدد الكلمات التقريبية}}
\]

\[
\text{Truncation Rate}=\frac{\text{النصوص التي تتجاوز الحد}}{\text{كل النصوص}}
\]

خصوبة أعلى تعني عادة تسلسلاً أطول وكلفة أكبر، لكنها ليست وحدها حكمًا على جودة النموذج.

In [14]:
def word_count(text: str) -> int:
    return max(1, len(text.split()))


def token_fertility(tokenizer: Tokenizer, text: str) -> float:
    tokens = tokenizer.encode(text).tokens
    content = [t for t in tokens if t not in {"[CLS]", "[SEP]", "[PAD]"}]
    return len(content) / word_count(text)


def truncation_rate(tokenizer: Tokenizer, texts: Iterable[str], max_length: int) -> float:
    texts = list(texts)
    if not texts:
        raise ValueError("texts must not be empty")
    return sum(len(tokenizer.encode(t).ids) > max_length for t in texts) / len(texts)

model_texts = [record.model_text for record in records]
fertilities = [token_fertility(local_tokenizer, text) for text in model_texts]
rate_at_10 = truncation_rate(local_tokenizer, model_texts, max_length=10)
print("fertility per sample:", [round(x, 2) for x in fertilities])
print("mean fertility:", round(float(np.mean(fertilities)), 2))
print("truncation rate @10:", f"{rate_at_10:.0%}")
assert all(x > 0 for x in fertilities)
assert 0.0 <= rate_at_10 <= 1.0
print("Tokenisation metrics=PASS")

fertility per sample: [1.17, 1.0, 1.5, 2.0, 1.14]
mean fertility: 1.36
truncation rate @10: 0%
Tokenisation metrics=PASS


## 3) Padding وTruncation ثم Embeddings

- **Padding:** إضافة `[PAD]` لتوحيد طول الدفعة؛ قناع الانتباه يضع 0 عند الحشو.
- **Truncation:** قص النص عند حد؛ قد يحذف المعلومة الحاسمة.
- **Token ID:** فهرس صحيح.
- **Embedding:** متجه أعداد حقيقية يُتعلم أثناء التدريب.

المتجهات التالية عشوائية للشرح فقط وليست تضمينات لغوية مدربة.

In [15]:
MAX_LENGTH = 12
local_tokenizer.enable_truncation(max_length=MAX_LENGTH)
local_tokenizer.enable_padding(length=MAX_LENGTH, pad_id=vocab["[PAD]"], pad_token="[PAD]")
batch = [local_tokenizer.encode(text) for text in model_texts[:2]]
input_ids = np.array([item.ids for item in batch], dtype=np.int64)
attention_mask = np.array([item.attention_mask for item in batch], dtype=np.int64)
embedding_table = rng.normal(0.0, 0.02, size=(len(vocab), 8))
embeddings = embedding_table[input_ids]
print("input_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("embeddings shape:", embeddings.shape)
print("first attention mask:", attention_mask[0].tolist())
assert input_ids.shape == (2, MAX_LENGTH)
assert attention_mask.shape == input_ids.shape
assert embeddings.shape == (2, MAX_LENGTH, 8)
print("IDs-to-embeddings pipeline=PASS")

input_ids shape: (2, 12)
attention_mask shape: (2, 12)
embeddings shape: (2, 12, 8)
first attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
IDs-to-embeddings pipeline=PASS


**المتوقع / Expected:** الأشكال `(2, 12)` و`(2, 12, 8)` ثم رسالة نجاح. الحشو لا يحمل معنى؛ قناع الانتباه يمنع النموذج من معاملته كنص.

## Explore — مقارنة اختيارية مع mBERT

هذه الخلية تحاول تنزيل مرمّز `google-bert/bert-base-multilingual-cased` من Hugging Face. الاستخدام مجاني، لكن التنزيل يحتاج شبكة وقد يتعذر مؤقتًا. فشلها **لا يمنع اجتياز Core**.

In [16]:
try:
    from transformers import AutoTokenizer
    hf_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-multilingual-cased", use_fast=True)
    comparison_text = "معالجة اللغة الطبيعية مفيدة Natural language processing is useful"
    hf_tokens = hf_tokenizer.tokenize(comparison_text)
    print("mBERT tokens:", hf_tokens)
    print("mBERT token count:", len(hf_tokens))
    print("Fast tokenizer:", hf_tokenizer.is_fast)
except Exception as exc:
    print("Optional mBERT download unavailable; continue with Core.")
    print("Reason:", type(exc).__name__)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

mBERT tokens: ['مع', '##الجة', 'اللغة', 'الطبيعية', 'م', '##فيد', '##ة', 'Natural', 'language', 'processing', 'is', 'useful']
mBERT token count: 12
Fast tokenizer: True


## 4) قرار موثق / Documented decision

| البند / Criterion | ملاحظتي / My evidence |
|---|---|
| Profile المطبق |  |
| مثال قبل/بعد |  |
| متوسط Token Fertility |  |
| Truncation Rate والحد |  |
| خطر محتمل على العربية |  |
| قراري والسبب |  |

### مستويات التحدي

- **Core:** شغّل Core واشرح مثالًا قبل/بعد.
- **Explore:** قارن المحلي مع mBERT على 5 جمل اصطناعية.
- **Distinction:** قارن مرمزين مناسبين وناقش الخصوبة والقطع والزمن دون اعتبار مقياس واحد إثباتًا للأفضلية.

In [17]:
core_checks = {
    "unicode": unicode_rows[0]["code_point"] == "U+0623",
    "spacy_sentence_pipeline": all(len(value) == 2 for value in sentence_splits.values()),
    "raw_copy_preserved": records[2].raw_text == samples[2],
    "pii_masked": "<EMAIL>" in records[2].model_text and "<PHONE>" in records[3].model_text,
    "token_metrics": all(x > 0 for x in fertilities) and 0 <= rate_at_10 <= 1,
    "embedding_shape": embeddings.shape == (2, MAX_LENGTH, 8),
}
for name, passed in core_checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")
assert all(core_checks.values())
print("DAY1_NOTEBOOK1_CORE=PASS")

unicode: PASS
spacy_sentence_pipeline: PASS
raw_copy_preserved: PASS
pii_masked: PASS
token_metrics: PASS
embedding_shape: PASS
DAY1_NOTEBOOK1_CORE=PASS


## نقطة GitHub / GitHub checkpoint

1. من Colab: **File → Save a copy in GitHub**.
2. احفظ النسخة في مستودعك العام داخل `notebooks/`.
3. Commit: `feat(day1): complete preprocessing and tokenisation lab`.
4. أضف في `README.md`: Profile، الخصوبة، معدل القطع، وقرارك.
5. لا تحفظ مفاتيح أو بيانات حقيقية أو مخرجات تكشف معلومات شخصية.

**دليل الاكتمال:** `DAY1_NOTEBOOK1_CORE=PASS` + رابط Commit عام + جدول القرار.

In [25]:
# Distinction — مقارنة ثنائية اللغة قابلة لإعادة التشغيل من الصفر

import time
import numpy as np
from transformers import AutoTokenizer

DISTINCTION_SEEDS = [7, 42, 2026]

arabic_evaluation = [
    "وبالخدمة الإلكترونية الجديدة تتحسن تجربة المستفيد.",
    "يتطلب التحقق من المعطيات استيثاقاً قبل اعتماد الاستنتاج.",
    "تساعد المعالجة اللغوية في تحليل الملاحظات التعليمية.",
    "يجب الحفاظ على خصوصية البيانات أثناء بناء النظام.",
]

english_evaluation = [
    "The electronic service improves the learner experience.",
    "Evidence should be verified before accepting a conclusion.",
    "Natural language processing can analyse educational feedback.",
    "Privacy must be preserved while building the system.",
]

bilingual_corpus = arabic_evaluation + english_evaluation


def measured_fertility(tokenizer, texts):
    values = []

    for text in texts:
        words = max(1, len(text.split()))
        pieces = len(tokenizer.tokenize(text))
        values.append(pieces / words)

    return float(np.mean(values))


def measured_truncation_rate(tokenizer, texts, max_length):
    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
    truncated = 0

    for text in texts:
        token_count = len(tokenizer.tokenize(text)) + special_tokens

        if token_count > max_length:
            truncated += 1

    return truncated / len(texts)


def measured_tokenization_time_ms(tokenizer, texts, repeats=20):
    start = time.perf_counter()

    for _ in range(repeats):
        for text in texts:
            tokenizer.tokenize(text)

    elapsed = time.perf_counter() - start
    calls = repeats * len(texts)

    return (elapsed / calls) * 1000


try:
    mbert_tokenizer = AutoTokenizer.from_pretrained(
        "google-bert/bert-base-multilingual-cased",
        use_fast=True,
    )

    arabert_tokenizer = AutoTokenizer.from_pretrained(
        "aubmindlab/bert-base-arabertv02",
        use_fast=True,
    )

    tokenizers_to_compare = {
        "mBERT": mbert_tokenizer,
        "AraBERT": arabert_tokenizer,
    }

    print("=== الخصوبة الثنائية اللغة ===")

    for name, tokenizer in tokenizers_to_compare.items():
        ar_fertility = measured_fertility(
            tokenizer,
            arabic_evaluation,
        )

        en_fertility = measured_fertility(
            tokenizer,
            english_evaluation,
        )

        mean_time = measured_tokenization_time_ms(
            tokenizer,
            bilingual_corpus,
        )

        print(f"\n{name}")
        print(f"Arabic fertility:  {ar_fertility:.3f}")
        print(f"English fertility: {en_fertility:.3f}")
        print(f"Mean tokenisation time: {mean_time:.4f} ms")

    print("\n=== معدل القطع ===")

    for name, tokenizer in tokenizers_to_compare.items():
        print(f"\n{name}")

        for max_length in [32, 64]:
            ar_rate = measured_truncation_rate(
                tokenizer,
                arabic_evaluation,
                max_length,
            )

            en_rate = measured_truncation_rate(
                tokenizer,
                english_evaluation,
                max_length,
            )

            combined_rate = measured_truncation_rate(
                tokenizer,
                bilingual_corpus,
                max_length,
            )

            print(f"max_length = {max_length}")
            print(f"Arabic:  {ar_rate:.1%}")
            print(f"English: {en_rate:.1%}")
            print(f"Combined: {combined_rate:.1%}")

    print("\n=== اختبار اللصيقة العربية ===")

    for text in ["الخدمة", "وبالخدمة"]:
        pieces = mbert_tokenizer.tokenize(text)

        print(text)
        print(pieces)
        print("piece count:", len(pieces))

    assert len(
        mbert_tokenizer.tokenize("وبالخدمة")
    ) >= len(
        mbert_tokenizer.tokenize("الخدمة")
    )

    print("DISTINCTION_CLITIC_TEST=PASS")

    print("\n=== تجربة ثلاث بذور للتضمينات التعليمية ===")

    example_ids = np.array(
        [
            [2, 5, 6, 7, 3],
            [2, 8, 9, 3, 0],
        ],
        dtype=np.int64,
    )

    vocab_size = 16
    embedding_dim = 8

    seed_outputs = {}

    for seed in DISTINCTION_SEEDS:
        seed_rng = np.random.default_rng(seed)

        embedding_table = seed_rng.normal(
            0.0,
            0.02,
            size=(vocab_size, embedding_dim),
        )

        embedded = embedding_table[example_ids]
        seed_outputs[seed] = embedded

        print(
            f"seed={seed}",
            "shape=",
            embedded.shape,
            "first_vector=",
            np.round(embedded[0, 0], 6).tolist(),
        )

    assert all(
        output.shape == (2, 5, 8)
        for output in seed_outputs.values()
    )

    assert not np.allclose(
        seed_outputs[7],
        seed_outputs[42],
    )

    assert not np.allclose(
        seed_outputs[42],
        seed_outputs[2026],
    )

    print("DISTINCTION_THREE_SEEDS=PASS")
    print("DISTINCTION_TOKENIZER_COMPARISON=PASS")
    print("BILINGUAL_TRUNCATION_MEASUREMENT=PASS")
    print("DAY1_DISTINCTION_REPRODUCIBLE=PASS")

except Exception as exc:
    print(
        "Distinction model comparison could not run because "
        "the external model download was unavailable."
    )
    print("Reason:", type(exc).__name__, str(exc))

=== الخصوبة الثنائية اللغة ===

mBERT
Arabic fertility:  2.595
English fertility: 1.299
Mean tokenisation time: 0.0782 ms

AraBERT
Arabic fertility:  1.182
English fertility: 3.714
Mean tokenisation time: 0.0866 ms

=== معدل القطع ===

mBERT
max_length = 32
Arabic:  0.0%
English: 0.0%
Combined: 0.0%
max_length = 64
Arabic:  0.0%
English: 0.0%
Combined: 0.0%

AraBERT
max_length = 32
Arabic:  0.0%
English: 0.0%
Combined: 0.0%
max_length = 64
Arabic:  0.0%
English: 0.0%
Combined: 0.0%

=== اختبار اللصيقة العربية ===
الخدمة
['ال', '##خدمة']
piece count: 2
وبالخدمة
['وب', '##ال', '##خدمة']
piece count: 3
DISTINCTION_CLITIC_TEST=PASS

=== تجربة ثلاث بذور للتضمينات التعليمية ===
seed=7 shape= (2, 5, 8) first_vector= [-0.026884, -0.009152, -0.038024, -0.025791, -0.036835, -0.004702, -0.025349, 0.005425]
seed=42 shape= (2, 5, 8) first_vector= [0.007375, -0.019178, 0.017569, -0.000999, -0.003697, -0.013619, 0.024451, -0.003091]
seed=2026 shape= (2, 5, 8) first_vector= [-0.008075, 0.010965, -0.00